# Помесячная статистика портфеля с НВВ

Ноутбук:
- находит все даты по колонкам `OD_{дата}`;
- для каждого месяца использует `OD`, `НИ`, `ПФН`, `НВВ`, `рестра`, `Обеспеченность`;
- валюту договора берет из постоянного столбца `Валюта договора` / `Валюта` / `Код валюты`;
- оставляет разбивку **Сегмент → Тип операции (Актив/УО)**;
- распределяет OD по 8 взаимоисключающим группам;
- создает отдельный DataFrame на каждую дату в `summary_by_date`;
- сохраняет каждую дату на отдельный лист Excel через `openpyxl`.

## Приоритет групп

1. Рестра
2. ПФН + необеспеченный
3. ПФН + остальная обеспеченность
4. НВВ — только Актив, валюта не BYN, без НИ и ПФН
5. НИ + необеспеченный
6. НИ + остальная обеспеченность
7. Только необеспеченный, без НИ и ПФН
8. Без НИ и ПФН + остальная обеспеченность

Для **УО** признак НВВ в группах 7–8 игнорируется.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 1. Настройки

In [ ]:
INPUT_FILE = Path("portfolio_monthly.xlsx")
OUTPUT_FILE = Path("Статистика_помесячно_НВВ.xlsx")

SHEET_NAME = 0

COL_SEGMENT = "Сегмент"
COL_OPERATION = "Тип операции"

# Возможные названия постоянной колонки с валютой договора
CURRENCY_COLUMN_CANDIDATES = [
    "Валюта договора",
    "Валюта",
    "Код валюты",
]

# Значения, которые считаем BYN
BYN_VALUES = {
    "BYN",
    "933",
    "БЕЛОРУССКИЙ РУБЛЬ",
    "БЕЛОРУБ",
}

# Если OD в тыс. BYN, на выходе будет млн BYN
DIVIDE_OD_BY_1000 = True


## 2. Вспомогательные функции

In [ ]:
def prepare_flag(series):
    s = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(",", ".", regex=False)
    )

    mapping = {
        "1": 1, "1.0": 1, "да": 1, "yes": 1, "true": 1,
        "0": 0, "0.0": 0, "нет": 0, "no": 0, "false": 0,
        "nan": 0, "none": 0, "": 0,
    }

    result = s.map(mapping)
    numeric = pd.to_numeric(s, errors="coerce")
    return result.fillna(numeric)


def prepare_number(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0)

    s = (
        series.astype(str)
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    return pd.to_numeric(s, errors="coerce").fillna(0)


def normalize_name(value):
    return (
        str(value)
        .replace("\xa0", " ")
        .strip()
        .lower()
        .replace("ё", "е")
    )


def find_month_column(df, prefix, date):
    target = normalize_name(f"{prefix}_{date}")

    for col in df.columns:
        if normalize_name(col) == target:
            return col

    return None


def find_any_column(df, candidates):
    normalized = {
        normalize_name(col): col
        for col in df.columns
    }

    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]

    raise KeyError(
        "Не найдена колонка. Ожидался один из вариантов: "
        f"{candidates}\n\nКолонки файла:\n{list(df.columns)}"
    )


def normalize_operation(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("ё", "е")
    )


def normalize_currency(series):
    return (
        series.fillna("")
        .astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.strip()
        .str.upper()
    )


## 3. Чтение файла

In [ ]:
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

COL_CURRENCY = find_any_column(
    df,
    CURRENCY_COLUMN_CANDIDATES
)

print(f"Строк: {len(df):,}")
print(f"Колонок: {len(df.columns):,}")
print(f"Колонка валюты: {COL_CURRENCY}")

display(df.head())


## 4. Находим все даты по колонкам OD_*

In [ ]:
dates = []

for col in df.columns:
    col_str = str(col).strip()

    if col_str.lower().startswith("od_"):
        dates.append(col_str[3:])

dates = list(dict.fromkeys(dates))

print("Найденные даты:")
for date in dates:
    print(date)

print(f"\nВсего дат: {len(dates)}")


## 5. Критерии

In [ ]:
C1 = "1. Рестра"
C2 = "2. ПФН + необеспеченный"
C3 = "3. ПФН + остальная обеспеченность"
C4 = "4. НВВ (Актив, валюта != BYN)"
C5 = "5. НИ + необеспеченный"
C6 = "6. НИ + остальная обеспеченность"
C7 = "7. Только необеспеченный"
C8 = "8. Без НИ/ПФН + остальная обеспеченность"

criteria_order = [
    C1, C2, C3, C4,
    C5, C6, C7, C8,
]


## 6. Создаем отдельный DataFrame для каждой даты

Результаты будут доступны так:

```python
summary_by_date["31.01.2026"]
summary_by_date["28.02.2026"]
```


In [ ]:
summary_by_date = {}
raw_by_date = {}
skipped_dates = {}

for date in dates:

    col_od = find_month_column(df, "OD", date)
    col_ni = find_month_column(df, "НИ", date)
    col_pfn = find_month_column(df, "ПФН", date)
    col_nvv = find_month_column(df, "НВВ", date)
    col_restra = find_month_column(df, "рестра", date)
    col_collateral = find_month_column(df, "Обеспеченность", date)

    month_cols = {
        "OD": col_od,
        "НИ": col_ni,
        "ПФН": col_pfn,
        "НВВ": col_nvv,
        "рестра": col_restra,
        "Обеспеченность": col_collateral,
    }

    missing = [
        name
        for name, col in month_cols.items()
        if col is None
    ]

    if missing:
        skipped_dates[date] = missing
        print(f"Пропуск {date}: нет колонок {missing}")
        continue

    tmp = pd.DataFrame({
        "Сегмент": df[COL_SEGMENT],
        "Тип операции": df[COL_OPERATION],
        "Валюта": df[COL_CURRENCY],
        "Дата": date,
    })

    tmp["OD"] = prepare_number(df[col_od])

    if DIVIDE_OD_BY_1000:
        tmp["OD"] = tmp["OD"] / 1000

    tmp["НИ"] = prepare_flag(df[col_ni])
    tmp["ПФН"] = prepare_flag(df[col_pfn])
    tmp["НВВ"] = prepare_flag(df[col_nvv])
    tmp["Рестра"] = prepare_flag(df[col_restra])

    # --------------------------------------------------------
    # Обеспеченность:
    # только два состояния для этой разбивки:
    # Необеспеченный / Остальная обеспеченность.
    # --------------------------------------------------------

    collateral = (
        df[col_collateral]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("ё", "е")
    )

    # "Недостаточно обеспеченный" НЕ считается необеспеченным.
    tmp["Необеспеченный"] = (
        collateral.str.contains(
            r"\bне\s*обеспеч|необеспеч",
            regex=True,
            na=False,
        )
        & ~collateral.str.contains(
            "недостаточно",
            regex=False,
            na=False,
        )
    )

    tmp["Остальная обеспеченность"] = ~tmp["Необеспеченный"]

    operation_norm = normalize_operation(tmp["Тип операции"])
    currency_norm = normalize_currency(tmp["Валюта"])

    tmp["_Актив"] = operation_norm.eq("актив")
    tmp["_УО"] = operation_norm.eq("уо")
    tmp["_BYN"] = currency_norm.isin(BYN_VALUES)
    tmp["_Валютный"] = ~tmp["_BYN"]

    # --------------------------------------------------------
    # 8 взаимоисключающих групп.
    #
    # Приоритет:
    # Рестра -> ПФН -> НВВ -> НИ -> обеспеченность.
    #
    # Если НИ и ПФН одновременно = 1, запись относится к ПФН.
    # Для УО НВВ в группах 7-8 не имеет значения.
    # --------------------------------------------------------

    conditions = [
        # 1. Рестра
        tmp["Рестра"].eq(1),

        # 2. ПФН + необеспеченный
        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(1)
            & tmp["Необеспеченный"]
        ),

        # 3. ПФН + остальная обеспеченность
        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(1)
            & tmp["Остальная обеспеченность"]
        ),

        # 4. Только НВВ:
        # Актив, валюта не BYN, без НИ и ПФН.
        # Обеспеченность любая.
        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(0)
            & tmp["НИ"].eq(0)
            & tmp["НВВ"].eq(1)
            & tmp["_Актив"]
            & tmp["_Валютный"]
        ),

        # 5. НИ + необеспеченный, без ПФН
        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(0)
            & tmp["НИ"].eq(1)
            & tmp["Необеспеченный"]
        ),

        # 6. НИ + остальная обеспеченность, без ПФН
        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(0)
            & tmp["НИ"].eq(1)
            & tmp["Остальная обеспеченность"]
        ),

        # 7. Только необеспеченный, без НИ и ПФН.
        # Для УО НВВ намеренно не проверяем.
        # Для Активов НВВ в валюте уже был пойман группой 4.
        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(0)
            & tmp["НИ"].eq(0)
            & tmp["Необеспеченный"]
        ),

        # 8. Без НИ/ПФН + остальная обеспеченность.
        # Для УО НВВ также намеренно игнорируем.
        (
            tmp["Рестра"].eq(0)
            & tmp["ПФН"].eq(0)
            & tmp["НИ"].eq(0)
            & tmp["Остальная обеспеченность"]
        ),
    ]

    choices = [
        C1, C2, C3, C4,
        C5, C6, C7, C8,
    ]

    tmp["Критерий"] = np.select(
        conditions,
        choices,
        default="НЕ РАСПРЕДЕЛЕНО"
    )

    # Только Актив и УО
    tmp = tmp[
        tmp["_Актив"] | tmp["_УО"]
    ].copy()

    raw_by_date[date] = tmp

    summary_date = (
        tmp[
            tmp["Критерий"] != "НЕ РАСПРЕДЕЛЕНО"
        ]
        .pivot_table(
            index=[
                "Сегмент",
                "Тип операции",
            ],
            columns="Критерий",
            values="OD",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(
            columns=criteria_order,
            fill_value=0
        )
        .reset_index()
    )

    summary_date["ИТОГО"] = (
        summary_date[criteria_order]
        .sum(axis=1)
    )

    summary_by_date[date] = summary_date


print(
    f"Сформировано DataFrame: "
    f"{len(summary_by_date)}"
)

if skipped_dates:
    print("\nПропущенные даты:")
    for date, missing in skipped_dates.items():
        print(f"{date}: {missing}")


## 7. Список созданных DataFrame

In [ ]:
list(summary_by_date.keys())


## 8. Вывести все даты

In [ ]:
for date, df_date in summary_by_date.items():
    print(f"\n===== {date} =====")
    display(df_date)


## 9. Получить конкретную дату

Например:

```python
df_jan = summary_by_date["31.01.2026"]
df_jan
```


In [ ]:
# Автоматически покажем первую найденную дату
if summary_by_date:
    first_date = next(iter(summary_by_date))
    print("Первая дата:", first_date)
    display(summary_by_date[first_date])


## 10. Сохранить каждый месяц на отдельный лист Excel

In [ ]:
with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    for date, df_date in summary_by_date.items():

        sheet_name = str(date)

        # Недопустимые символы Excel
        for ch in ["/", "\\", ":", "*", "?", "[", "]"]:
            sheet_name = sheet_name.replace(ch, ".")

        sheet_name = sheet_name[:31]

        df_date.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        ws = writer.book[sheet_name]

        # Закрепляем заголовок и первые две колонки
        ws.freeze_panes = "C2"

        # Простая ширина колонок без xlsxwriter
        ws.column_dimensions["A"].width = 20
        ws.column_dimensions["B"].width = 18

        for col_idx in range(3, len(df_date.columns) + 1):
            from openpyxl.utils import get_column_letter
            ws.column_dimensions[
                get_column_letter(col_idx)
            ].width = 24


print(f"Готово: {OUTPUT_FILE.resolve()}")
